# MLP Capacity Profiling

Profile how MLPs use their capacity: dead neurons, activation diversity,
weight utilization, and information throughput.

## Why This Matters

MLP capacity provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_capacity_profiling import (
    mlp_dead_neuron_profile, mlp_activation_diversity,
    mlp_weight_utilization, mlp_information_throughput,
    mlp_capacity_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Dead Neuron Profile

How many MLP neurons are effectively dead (very low activation)?

In [ ]:
result = mlp_dead_neuron_profile(model, tokens, layer=0)
print(f"Dead: {result['n_dead']}/{result['n_total']} ({result['dead_fraction']:.1%})")
print(f"Mean activity: {result['mean_activity']:.6f}")
if result['dead_neuron_ids']:
    print(f"Dead neuron IDs (first 20): {result['dead_neuron_ids']}")

## Activation Diversity

How diverse are activations across positions?

In [ ]:
result = mlp_activation_diversity(model, tokens, layer=0)
print(f"Mean diversity: {result['mean_diversity']:.4f}")
print(f"\nMost diverse neurons:")
for idx, score in result['most_diverse']:
    bar = '█' * int(score * 30)
    print(f"  Neuron {idx}: {score:.4f} {bar}")
print(f"\nLeast diverse neurons:")
for idx, score in result['least_diverse']:
    bar = '█' * int(score * 30)
    print(f"  Neuron {idx}: {score:.4f} {bar}")

## Weight Utilization

How much of the MLP weight matrix capacity is used?

In [ ]:
result = mlp_weight_utilization(model, layer=0)
print(f"W_in effective rank: {result['w_in_effective_rank']:.2f} "
      f"(utilization: {result['w_in_utilization']:.3f})")
print(f"W_out effective rank: {result['w_out_effective_rank']:.2f} "
      f"(utilization: {result['w_out_utilization']:.3f})")
print(f"Dimensions: d_model={result['d_model']}, d_mlp={result['d_mlp']}")

## Information Throughput

How much information passes through each MLP layer?

In [ ]:
result = mlp_information_throughput(model, tokens, layer=0)
print(f"Mean throughput ratio: {result['mean_throughput_ratio']:.4f}")
print(f"Mean residual fraction: {result['mean_residual_fraction']:.4f}")
for p in result['per_position']:
    bar = '█' * int(p['throughput_ratio'] * 10)
    print(f"  Pos {p['position']}: ratio={p['throughput_ratio']:.4f} {bar}")

## Capacity Summary Across Layers

In [ ]:
result = mlp_capacity_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: dead={p['dead_fraction']:.1%} "
          f"diversity={p['diversity_score']:.3f} "
          f"W_in_util={p['w_in_utilization']:.3f} "
          f"W_out_util={p['w_out_utilization']:.3f}")